# Modelo Final y Predicciones

En este notebook se entrena el mejor modelo identificado en los experimentos
y se realizan predicciones sobre 5 datos diferentes.

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [2]:
# Cargar datos
df = pd.read_csv('../Data/video_game_reviews.csv')

print(f"Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas")
df.head()

Dataset cargado: 47774 registros, 18 columnas


,Game Title,User Rating,Age Group Targeted,Price,Platform,Requires Special Device,Developer,Publisher,Release Year,Genre,Multiplayer,Game Length (Hours),Graphics Quality,Soundtrack Quality,Story Quality,User Review Text,Game Mode,Min Number of Players
0,Grand Theft Auto V,36.4,All Ages,41.41,PC,No,Game Freak,Innersloth,2015,Adventure,No,55.3,Medium,Average,Poor,"Solid game, but too many bugs.",Offline,1
1,The Sims 4,38.3,Adults,57.56,PC,No,Nintendo,Electronic Arts,2015,Shooter,Yes,34.6,Low,Poor,Poor,"Solid game, but too many bugs.",Offline,3
2,Minecraft,26.8,Teens,44.93,PC,Yes,Bungie,Capcom,2012,Adventure,Yes,13.9,Low,Good,Average,"Great game, but the graphics could be better.",Offline,5
3,Bioshock Infinite,38.4,All Ages,48.29,Mobile,Yes,Game Freak,Nintendo,2015,Sports,No,41.9,Medium,Good,Excellent,"Solid game, but the graphics could be better.",Online,4
4,Half-Life: Alyx,30.1,Adults,55.49,PlayStation,Yes,Game Freak,Epic Games,2022,RPG,Yes,13.2,High,Poor,Good,"Great game, but too many bugs.",Offline,1


In [3]:
# Preparar datos
target = 'User Rating'

# Separar features y target
X = df.drop(columns=[target])
y = df[target]

# Codificar variables categóricas
cat_cols = X.select_dtypes(include=['object', 'category']).columns
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Datos preparados:")
print(f"  Train: {X_train_scaled.shape}")
print(f"  Test: {X_test_scaled.shape}")

Datos preparados:
  Train: (38219, 99)
  Test: (9555, 99)


## Mejor Modelo Identificado

De acuerdo a los experimentos realizados, el mejor modelo fue:

**Regresión Lineal** con los siguientes hiperparámetros:
- `fit_intercept`: True
- `positive`: True

**Métricas obtenidas en experimentación:**
- MAE: 1.0007
- RMSE: 1.1581

In [4]:
# Entrenar el mejor modelo con mejores hiperparámetros
best_model = LinearRegression(fit_intercept=True, positive=True)
best_model.fit(X_train_scaled, y_train)

# Evaluar en test
y_pred_test = best_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

print("="*60)
print("MÉTRICAS DEL MODELO FINAL EN TEST")
print("="*60)
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")
print("="*60)

MÉTRICAS DEL MODELO FINAL EN TEST
MAE (Mean Absolute Error): 1.0007
RMSE (Root Mean Squared Error): 1.1581
R² Score: 0.9768


## Selección de 5 Datos Muy Diferentes

Se seleccionan 5 juegos con características muy diferentes para evaluar
el comportamiento del modelo en distintos escenarios.

In [5]:
# Seleccionar 5 datos muy diferentes del conjunto de test
test_indices = X_test.index

# Criterios de diversidad:
selected_indices = [
    y_test.idxmax(),  # Rating más alto
    y_test.idxmin(),  # Rating más bajo  
    y_test.iloc[(y_test - y_test.median()).abs().argsort()[:1]].index[0],  # Rating mediano
    test_indices[100],  # Índice arbitrario 1
    test_indices[200]   # Índice arbitrario 2
]

# Asegurarse de que todos sean únicos
selected_indices = list(dict.fromkeys(selected_indices))
selected_indices = selected_indices[:5]

print("="*60)
print("DATOS SELECCIONADOS PARA PREDICCIÓN")
print("="*60)

# Mostrar información de los juegos seleccionados
for i, idx in enumerate(selected_indices, 1):
    game_info = df.loc[idx]
    print(f"\nJuego {i}: {game_info.get('Game Title', 'N/A')}")
    print(f"  Rating Real: {y_test.loc[idx]:.2f}")
    print(f"  Género: {game_info.get('Genre', 'N/A')}")
    print(f"  Plataforma: {game_info.get('Platform', 'N/A')}")

DATOS SELECCIONADOS PARA PREDICCIÓN

Juego 1: Hades
  Rating Real: 49.30
  Género: Shooter
  Plataforma: PlayStation

Juego 2: The Legend of Zelda: Breath of the Wild
  Rating Real: 10.30
  Género: Puzzle
  Plataforma: PC

Juego 3: Animal Crossing: New Horizons
  Rating Real: 29.70
  Género: Action
  Plataforma: PC

Juego 4: 1000-Piece Puzzle
  Rating Real: 42.60
  Género: Simulation
  Plataforma: Nintendo Switch

Juego 5: Fortnite
  Rating Real: 28.10
  Género: RPG
  Plataforma: Mobile


In [6]:
# Hacer predicciones para los 5 datos seleccionados
selected_X = X_encoded.loc[selected_indices]
selected_X_scaled = scaler.transform(selected_X)
selected_y_true = y_test.loc[selected_indices]

predictions = best_model.predict(selected_X_scaled)

# Crear DataFrame con resultados
results_df = pd.DataFrame({
    'Juego': [df.loc[idx, 'Game Title'] for idx in selected_indices],
    'Rating Real': selected_y_true.values,
    'Rating Predicho': predictions,
    'Error Absoluto': np.abs(selected_y_true.values - predictions)
})

print("\n" + "="*60)
print("RESULTADOS DE LAS PREDICCIONES")
print("="*60)
print(f"\n{results_df.to_string(index=False)}")

mae_selected = mean_absolute_error(selected_y_true, predictions)
print(f"\nMAE en estos 5 datos: {mae_selected:.4f}")
print("="*60)


RESULTADOS DE LAS PREDICCIONES

                                  Juego  Rating Real  Rating Predicho  Error Absoluto
                                  Hades         49.3        47.732462        1.567538
The Legend of Zelda: Breath of the Wild         10.3        11.896743        1.596743
          Animal Crossing: New Horizons         29.7        31.476845        1.776845
                      1000-Piece Puzzle         42.6        42.184956        0.415044
                               Fortnite         28.1        26.562094        1.537906

MAE en estos 5 datos: 1.3788


## Análisis de Resultados

### Desempeño del Modelo

El modelo de Regresión Lineal mostró un desempeño sólido:

1. **Error promedio bajo**: El MAE de aproximadamente 1.0 indica que, en promedio, las predicciones se desvían solo 1 punto del rating real.

2. **Alta capacidad explicativa**: El R² cercano a 0.97-0.98 indica que el modelo explica más del 97% de la variabilidad en los ratings.

3. **Generalización**: El modelo mantiene un rendimiento consistente en el conjunto de test.

### Predicciones en Datos Diversos

Las 5 predicciones realizadas sobre juegos con características muy diferentes muestran:

- El modelo maneja bien tanto ratings altos como bajos
- Las predicciones son razonables para diferentes géneros y plataformas
- Los errores son consistentes con las métricas generales del modelo

### Interpretación

La Regresión Lineal resultó ser el mejor modelo porque:
- Las relaciones en el dataset son predominantemente lineales
- Es computacionalmente eficiente
- Proporciona resultados interpretables